# 2. The transverse-field Ising chain as an MPO

**Goal.** Write the Hamiltonian $H = -J\sum_i X_iX_{i+1} - h\sum_i Z_i$ as a matrix product operator (MPO): a chain of small tensors, just like an MPS, but with two physical legs per site.

*Further reading:* Schollwöck [arXiv:1008.3477](https://arxiv.org/abs/1008.3477), Sec. 6.1; G. Crosswhite and D. Bacon, [arXiv:0708.1221](https://arxiv.org/abs/0708.1221) (MPOs as finite-state automata).

In [1]:
import inspect

import numpy as np

import dmrg

## Intuition: a little machine that writes the Hamiltonian

Read the chain from left to right and imagine a machine with three states:

* **0**: nothing placed yet (put identities and stay), 
* **1**: an $X$ was just placed, so the next site *must* put $-JX$ to finish the bond,
* **2**: a term is finished (put identities until the end).

The transitions form the $3\times 3$ operator-valued matrix

$$W = \begin{pmatrix} I & X & -hZ \\ 0 & 0 & -JX \\ 0 & 0 & I \end{pmatrix},$$

with row = state before the site and column = state after. Multiplying $W W \cdots W$ and keeping the $(0, 2)$ entry sums over every path from state 0 to state 2, and each path is exactly one term of $H$. The first site keeps only row 0 and the last only column 2.

So the TFIM needs MPO bond dimension 3, independent of $L$. In code `W[w_left, w_right, out, in]`:

In [2]:
print(inspect.getsource(dmrg.tfim_bulk_tensor))
print(inspect.getsource(dmrg.tfim_mpo))

def tfim_bulk_tensor(h, J=1.0):
    """The 3x3 bulk tensor W[w_left, w_right, out, in]."""
    W = np.zeros((3, 3, 2, 2))
    W[0, 0] = I2
    W[0, 1] = X
    W[0, 2] = -h * Z
    W[1, 2] = -J * X
    W[2, 2] = I2
    return W

def tfim_mpo(L, h, J=1.0):
    """List of L MPO tensors; the first keeps row 0 and the last keeps column 2."""
    W = tfim_bulk_tensor(h, J)
    mpo = [W.copy() for _ in range(L)]
    mpo[0] = W[0:1, :, :, :]  # left boundary: start in state 0
    mpo[-1] = W[:, 2:3, :, :]  # right boundary: end in state 2
    return mpo



## Check against the obvious construction

Build $H$ with Kronecker products for a few spins and compare.

In [3]:
X = np.array([[0, 1], [1, 0]])
Z = np.diag([1, -1])


def dense_tfim(L, h):
    def op(o, i):  # o acting on site i
        return np.kron(np.kron(np.eye(2**i), o), np.eye(2 ** (L - i - 1)))

    return -sum(op(X, i) @ op(X, i + 1) for i in range(L - 1)) - h * sum(op(Z, i) for i in range(L))


for L in [2, 4, 7]:
    print(L, np.allclose(dmrg.mpo_to_dense(dmrg.tfim_mpo(L, 0.8)), dense_tfim(L, 0.8)))

2 True
4 True
7 True


## Energies without building $H$

$\langle\psi|H|\psi\rangle$ is a "sandwich": MPS on top, MPO in the middle, conjugate MPS at the bottom, contracted site by site from the left. The running object `E[x, w, y]` (ket bond, MPO bond, bra bond) is exactly the **left environment** that DMRG uses in notebook 3. For the all-up product state $|00\cdots0\rangle$ every $Z = +1$ and every $\langle XX\rangle = 0$, so $E = -hL$:

In [4]:
L, h = 50, 0.8
up = dmrg.product_mps([[1, 0]] * L)
print(dmrg.expectation(up, dmrg.tfim_mpo(L, h)), "vs", -h * L)

-39.999999999999986 vs -40.0


## The exact answer we will compare against

The Jordan–Wigner transformation maps the TFIM to free fermions (Lieb, Schultz, Mattis 1961; Pfeuty 1970). For the open chain, the single-particle energies are $2s_n$ with $s_n$ the singular values of the bidiagonal $L\times L$ matrix with $h$ on the diagonal and $J$ above it, and $E_0 = -\sum_n s_n$. For the infinite chain, $e_0 = -\frac1\pi\int_0^\pi \sqrt{J^2 + h^2 - 2Jh\cos k}\,dk$, which is $-4/\pi$ at the critical point $h = J$.

In [5]:
for L in [4, 8, 12]:
    E_ed = np.linalg.eigvalsh(dmrg.mpo_to_dense(dmrg.tfim_mpo(L, 1.0)))[0]
    print(f"L = {L:2d}: exact diagonalization {E_ed:.12f}, free fermions {dmrg.tfim_open_energy(L, 1.0):.12f}")
print("infinite chain at h = 1:", dmrg.tfim_infinite_energy_density(1.0), "  -4/pi =", -4 / np.pi)

L =  4: exact diagonalization -4.758770483144, free fermions -4.758770483144
L =  8: exact diagonalization -9.837951447459, free fermions -9.837951447459


L = 12: exact diagonalization -14.925971109909, free fermions -14.925971109909
infinite chain at h = 1: -1.2732395447351625   -4/pi = -1.2732395447351628


For larger $L$ the free-fermion formula costs only an $L\times L$ SVD, so it checks DMRG at any size.

## Exercises

1. Add a longitudinal field $-g\sum_i X_i$. Which entry of $W$ changes?
2. Write the MPO of the Heisenberg chain $\sum_i \vec S_i\cdot\vec S_{i+1}$. What bond dimension does it need? (Answer: 5.)
3. Write the MPO of $\sum_{i<j} e^{-(j-i)/\xi} X_iX_j$ with bond dimension 3. Hint: state 1 can "decay" by staying in state 1 with weight $e^{-1/\xi}$.